# Analyzing CSKG

This notebook performs various analyses on CSKG

Parameters are set up in the first cell so that we can run this notebook in batch mode. Example invocation command:

```
papermill Example8\ -\ Wikidata\ Subset.ipynb example8.out.ipynb \
-p cskg_path /Users/pedroszekely/Downloads/kypher/cskg \
-p kg cskg_connected.tsv.gz \
-p delete_database no 
```

### Parameters for invoking the notebook

- `cskg_path`: a folder containing the CSKG edges file and all the analysis products.
- `kg`: the name of the edge file.
- `delete_database`: whether to delete the SQL database before running the notebook: "" or "no" means don't delete it.

# Preamble

Set up paths and environment variables

In [ ]:
# Parameters
cskg_path = "/Users/pedroszekely/Downloads/kypher/cskg"
kg = "cskg_connected.kgtk.gz"
delete_database = "yes"

In [ ]:
import io
import os
import subprocess
import sys

import numpy as np
import pandas as pd

import altair as alt

In [ ]:
os.environ['CSKG'] = cskg_path
os.environ['KG'] = "{}/{}".format(cskg_path, kg)
os.environ['NKG'] = "{}/cskg-normalized.kgtk.gz".format(cskg_path, kg)
os.environ['STORE'] = "{}/wikidata.sqlite3.db".format(cskg_path)
os.environ['kypher'] = "time kgtk query --graph-cache " + os.environ['STORE']
# os.environ['kypher'] = "time kgtk --debug query --graph-cache " + os.environ['STORE']

In [ ]:
!echo $CSKG
!echo $KG
!echo $NKG
!echo $kypher
!echo $STORE

In [ ]:
cd $cskg_path

In [ ]:
if delete_database and delete_database != "no":
    print("Deleted database")
    !rm $STORE

# Utilities

In [ ]:
def bar_chart(data, x_column, y_column, title="", width=800):
    """Construct a simple bar chart with two properties"""
    bars = alt.Chart(data).mark_bar().encode(
        y=alt.Y(y_column, sort='-x'),
        x=x_column
    ).properties(
        title=title,
        width=width
    )

    text = bars.mark_text(
        align='left',
        baseline='middle',
        dx=3  # Nudges text to right so it doesn't appear on top of the bar
    ).encode(
        text=x_column
    )

    return (bars + text)

In [ ]:
import io
import pandas
import subprocess

def shell_df(command, shell=False, **kwargs):
    """
    Takes a shell command as a string and and reads the result into a Pandas DataFrame.
    
    Additional keyword arguments are passed through to pandas.read_csv.
    
    :param command: a shell command that returns tabular data
    :type command: str
    :param shell: passed to subprocess.Popen
    :type shell: bool
    
    :return: a pandas dataframe
    :rtype: :class:`pandas.dataframe`
    """
    proc = subprocess.Popen(command, 
                            shell=shell,
                            stdout=subprocess.PIPE, 
                            stderr=subprocess.PIPE)
    output, error = proc.communicate()
    
    if proc.returncode == 0:
        if error:
            print(error.decode())
        with io.StringIO(output.decode()) as buffer:
            return pandas.read_csv(buffer, **kwargs)
    else:
        message = ("Shell command returned non-zero exit status: {0}\n\n"
                   "Command was:\n{1}\n\n"
                   "Standard error was:\n{2}")
        raise IOError(message.format(proc.returncode, command, error.decode()))

# Poking around

Print some lines to see what we have

In [ ]:
!zcat < "$KG" | head | column -t -s $'\t' 

Normalize the file so that it is easier to process with Kypher

In [ ]:
!kgtk normalize --verbose -i $KG -o $CSKG/temp.cskg.normalize.1.kgtk.gz --columns-to-lower 'relation;dimension' source sentence 'node1;label' 'relation;label' 'node2;label'

In [ ]:
!zcat < $CSKG/temp.cskg.normalize.1.kgtk.gz | head | column -t -s $'\t' 

Rename the columns to the standard `node1/label/node2` and add ids

In [ ]:
!kgtk rename-columns --mode NONE -i $CSKG/temp.cskg.normalize.1.kgtk.gz --output-columns id node1 label node2 \
/ add-id --id-style node1-label-node2 -o $NKG

Count the number of edges and nodes

In [ ]:
!$kypher -i $NKG \
--match '(n1)-[e]->(n2)' \
--return 'count(e) as num_edges, count(distinct n1) as num_nodes, count(distinct e.label) as num_relations, count(distinct n2) as num_values' \
| column -t -s $'\t' 

# Some Statistics

In [ ]:
command = "$kypher -i $NKG \
--match '(n1)-[e]->(n2)' \
--return 'distinct e.label, count(distinct n1) as nodes' \
--order-by 'count(distinct n1) desc'"
stats = shell_df(command, shell=True, sep='\t')

In [ ]:
bar_chart(stats[:20], 'nodes', 'label', title="Relations in CSKG")

### Distribution of edges in each data source

In [ ]:
command = "$kypher -i $NKG \
--match '(n1)-[r]->(n2), (r)-[l:source]->(s)' \
--return 's as source, count(distinct r) as `count of relations`' \
--order-by 'count(distinct r) desc'"
data = shell_df(command, shell=True, sep='\t')
bar_chart(data, 'count of relations', 'source', title="CSKG Relation Counts")

Compute the distribuiton of relations in each data source

In [ ]:
command = "$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source:`\"SOURCE\"`)' \
--return 'label as relation, count(distinct n1) as count' \
--order-by 'count(distinct n1) desc'"
datasets = []
for source in ["CN", "WN", "AT", "VG", "FN", "WD", "RG"]:
    data = shell_df(command.replace("SOURCE", source), shell=True, sep='\t')
    datasets.append(data)

In [ ]:
bar_chart(datasets[0], 'count', 'relation', title="ConceptNet: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[1], 'count', 'relation', title="WordNet: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[2], 'count', 'relation', title="Atomic: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[3], 'count', 'relation', title="Visual Genome: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[4], 'count', 'relation', title= "FrameNet: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[5], 'count', 'relation', "Wikidata: Count Of Relations", width=200)

In [ ]:
bar_chart(datasets[6], 'count', 'relation', title="Roget: Count Of Relations", width=200)

### ConceptNet nodes that contain `catch` and `throw`

In [ ]:
catch = !$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source)' \
--where 'source = $s and n1 =~ ".*/catch/.*"' \
--return 'distinct n1 as node1' \
--spara s='CN' \
--limit 100

for n in catch[1:-1]:
    print("https://www.conceptnet.io"+n)

In [ ]:
throw = !$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source)' \
--where 'source in [$cn] and n1 =~ ".*/throw?\\b.*"' \
--return 'distinct n1 as node1' \
--spara cn='CN' --spara wd='WD' --spara vg='VG'\
--limit 100

for n in throw[1:-1]:
    print("https://www.conceptnet.io"+n)

### ConceptNet nodes that contain `dog`

In [ ]:
dogs = !$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source)' \
--where 'source in [$cn] and n1 =~ ".*/dogs?\\b.*"' \
--return 'distinct n1 as node1' \
--spara cn='CN' \
--limit 100

for n in dogs[1:-1]:
    print("https://www.conceptnet.io"+n)

In [ ]:
frisbee = !$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source)' \
--where 'source in [$cn] and n1 =~ ".*/frisbees?\\b.*"' \
--return 'distinct n1 as node1' \
--spara cn='CN' --spara wd='WD' --spara vg='VG'\
--limit 100

for n in frisbee[1:-1]:
    print("https://www.conceptnet.io"+n)

### Get all the nodes in ConceptNet, Wikidata-CS and VisualGenone that contain `frisbee`

In [ ]:
!$kypher -i $NKG \
--match '(n1)-[r {label: label}]->(n2), (r)-[:source]->(source), (n1)-[:label]->(n1_label)' \
--where 'source in [$cn, $vg, $wd] and n1 =~ ".*frisbees?.*"' \
--return 'distinct source as source, n1 as node1, n1_label as `node1 label`, label as relation, n2 as node2' \
--order-by 'source, n1' \
--spara cn='CN' --spara wd='WD' --spara vg='VG'\
-o $CSKG/frisbee.tsv

We have many edges that relate to `frisbee`

In [ ]:
!wc $CSKG/frisbee.tsv

In [ ]:
!head $CSKG/frisbee.tsv | column -t -s $'\t' 